In [ ]:
!pip install --no-cache-dir "numpy<2.0" torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install --no-cache-dir recbole ray kmeans-pytorch pandas pyyaml tensorboard

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
import recbole
print("RecBole OK")

In [ ]:
!cd

In [ ]:
import numpy as np
print(np.__version__)  # 1.26.x 로 나와야 정상

In [ ]:
import pandas as pd
import os

dataset_name = 'microlens100k'
os.makedirs(f'dataset/{dataset_name}', exist_ok=True)

In [ ]:
df = pd.read_csv('MicroLens-100k_pairs.csv')

In [ ]:
df = df.rename(columns={
    'user': 'user_id:token',
    'item': 'item_id:token',
    'timestamp': 'timestamp:float'
})

In [ ]:
df.to_csv(f'dataset/{dataset_name}/{dataset_name}.inter', sep='\t', index=False)
print("데이터 변환 완료!")

In [ ]:
yaml_content = """
USER_ID_FIELD: user_id
ITEM_ID_FIELD: item_id
TIME_FIELD: timestamp
load_col:
  inter: [user_id, item_id, timestamp]
ITEM_LIST_LENGTH_FIELD: item_length
LIST_SUFFIX: _list
MAX_ITEM_LIST_LENGTH: 50

eval_args:
  split: {'LS': 'valid_and_test'}
  group_by: user
  order: TO
  mode:
    valid: uni100
    test: full
metrics: ['Hit', 'NDCG']
topk: [10]
valid_metric: NDCG@10

train_neg_sample_args: ~

model: SASRec
loss_type: 'CE'
learning_rate: 1e-5
hidden_size: 2048
n_layers: 2
n_heads: 2
dropout_prob: 0.1
train_batch_size: 128
eval_batch_size: 128
epochs: 50
eval_step: 5
stopping_step: 5

state_tracker: ['Tensorboard']
checkpoint_dir: ./recbole_checkpoints/
"""

with open('sasrec_microlens.yaml', 'w') as f:
    f.write(yaml_content)
print("YAML 설정 파일 생성 완료!")

In [ ]:
%reload_ext tensorboard

%tensorboard --logdir=log_tensorboard/ --port=6006

In [ ]:
from recbole.quick_start import run_recbole

# 학습 및 평가 파이프라인 시작
result = run_recbole(model='SASRec', dataset='microlens100k', config_file_list=['sasrec_microlens.yaml'])

print("\n=== 최종 테스트 결과 ===")
print(result)

In [ ]:
from recbole.quick_start import run_recbole

# 이어서 학습할 체크포인트 파일 지정
config_dict = {
    'train_from': './recbole_checkpoints/SASRec-Apr-08-2026_20-37-34.pth'
}

print("🚀 39 Epoch부터 학습을 재개합니다...")

# 모델 실행
result = run_recbole(
    model='SASRec',
    dataset='microlens100k',
    config_file_list=['sasrec_microlens.yaml'],
    config_dict=config_dict
)

print("\n=== 최종 테스트 결과 ===")
print(result)